# CRNN-CTC on Google Colab (backup for Kaggle)

Before you start:
1. **Runtime menu -> Change runtime type -> Hardware accelerator = T4 GPU -> Save.**
2. **Left sidebar -> the key icon (Secrets) -> Add new secret:** name `GITHUB_TOKEN`, value = your token, and switch **Notebook access** on.
3. Run the cells in order, one at a time. **Do not use "Run all"**: cell 7 is the 2.5-hour full run.
4. Keep this browser tab open while a cell is running.

In [ ]:
# 1. GPU check: must print True and a GPU name
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# 2. get the code (private repo) and pick the branch
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
!git clone https://{token}@github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git
%cd Devanagari_Word_Recognition_Project
!git remote set-url origin https://github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git
!git checkout nikhil-crnn
!pip install -q rapidfuzz pytest pyyaml

In [ ]:
# 3. data: IIIT-HW-Dev Version 1 (about 1.8 GB, two archives), 8 parallel connections. Wait for DONE.
!python scripts/download_iiit.py --out /content/IIIT-HW-Hindi_v1.tar.gz
!mkdir -p data/raw/iiit_hw_dev
!tar -xzf /content/IIIT-HW-Hindi_v1.tar.gz -C data/raw/iiit_hw_dev
!tar -xzf data/raw/iiit_hw_dev/HindiSeg.tar.gz -C data/raw/iiit_hw_dev
!rm /content/IIIT-HW-Hindi_v1.tar.gz data/raw/iiit_hw_dev/HindiSeg.tar.gz

In [ ]:
# 4. check: expect 69853 / 12708 / 12869 and 0 missing
import csv, os
for split in ("train", "val", "test"):
    rows = list(csv.DictReader(open(f"data/processed/words/{split}.csv", encoding="utf-8")))
    missing = sum(not os.path.exists(r["image_path"]) for r in rows[:2000])
    print(split, len(rows), "rows | missing among the first 2000:", missing)

In [ ]:
# 5. unit tests (should all pass)
!python -m pytest -q

In [ ]:
# 6. SANITY RUN: must memorise 100 training images (CER -> 0)
!python -m src.crnn.train --config configs/crnn/overfit.yaml

In [ ]:
# 7. FULL RUN. Do not start before the sanity run passed. Time budget: max_minutes in the config.
!python -m src.crnn.train --config configs/crnn/full.yaml
# after a disconnect run the same command with --resume

In [ ]:
# 8. evaluate on the TEST set once; writes results/crnn_full_greedy/
!python -m src.crnn.evaluate --ckpt checkpoints/crnn_full/best.pt

In [ ]:
# 9. keep the results in your Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/p09_outputs
!cp -r checkpoints/crnn_full results/crnn_full_greedy /content/drive/MyDrive/p09_outputs/
!cat results/crnn_full_greedy/metrics.json